# Data exploration

E-commerce user behavior dataset — exploratory analysis before modeling.

In [ ]:
import pandas as pd
import numpy as np
import os
import math
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asifxzaman/e-commerce-behavior-dataset8000-users")

print("Path to dataset files:", path)

Path to dataset files: /Users/katia/.cache/kagglehub/datasets/asifxzaman/e-commerce-behavior-dataset8000-users/versions/1


In [ ]:
# List files in the dataset
print("Files in the dataset:")
for filename in os.listdir(path):
    print(filename)

Files in the dataset:
ecommerce_user_behavior_8000.csv


In [ ]:
# Load data
data = pd.read_csv(os.path.join(path, "ecommerce_user_behavior_8000.csv"))
print("Data shape:", data.shape)
print("Columns:", list(data.columns))

Data shape: (8000, 14)


## 1. Structure & quality

In [ ]:
print("First rows:")
display(data.head(10))

print("\nRandom sample:")
display(data.sample(5, random_state=42))

print("\nColumn types:")
data.info()

print("\nUnique values per column:")
print(data.nunique())

In [ ]:
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("Missing values:")
display(missing_df[missing_df["missing_count"] > 0])

rows_with_missing = data.isnull().any(axis=1).sum()
print(f"\nRows with at least one missing value: {rows_with_missing} ({rows_with_missing / len(data) * 100:.2f}%)")

plt.figure(figsize=(10, 4))
sns.heatmap(data.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing values pattern (yellow = missing)")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Duplicate rows: {data.duplicated().sum()}")
print(f"Duplicate user_id: {data['user_id'].duplicated().sum()}")

if data["user_id"].duplicated().any():
    dup_users = data[data["user_id"].duplicated(keep=False)].sort_values("user_id")
    print("\nSample duplicate user_ids:")
    display(dup_users.head(10))

## 2. Target variable (`purchase`)

In [ ]:
print("Purchase counts:")
print(data["purchase"].value_counts(dropna=False))

print("\nPurchase proportions:")
print(data["purchase"].value_counts(normalize=True, dropna=False).round(4))

plt.figure(figsize=(8, 5))
sns.countplot(x="purchase", data=data)
plt.title("Class balance: purchase vs no purchase")
plt.xlabel("purchase (1 = yes, 0 = no)")
plt.tight_layout()
plt.show()

## 3. Numerical features

In [ ]:
numeric_cols = data.select_dtypes(include=["number"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in ("user_id", "purchase")]

print("Numerical summary (excluding user_id):")
display(data[feature_cols].describe().T)

In [ ]:
n_cols = 3
n_rows = math.ceil(len(feature_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for i, col in enumerate(feature_cols):
    sns.histplot(data[col].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
def count_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = {col: count_outliers_iqr(data[col].dropna()) for col in feature_cols}
outlier_df = pd.Series(outlier_counts, name="outlier_count").sort_values(ascending=False)
print("Outlier counts (IQR rule):")
display(outlier_df.to_frame())

n_cols = 3
n_rows = math.ceil(len(feature_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(y=data[col].dropna(), ax=axes[i])
    axes[i].set_title(col)

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Categorical features

In [ ]:
cat_cols = ["gender", "device_type"]

for col in cat_cols:
    print(f"\n{col} value counts:")
    print(data[col].value_counts(dropna=False))

fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    sns.countplot(x=col, data=data, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 5. Categorical features vs purchase

In [ ]:
for col in cat_cols:
    print(f"\n{col} vs purchase (row proportions):")
    display(pd.crosstab(data[col], data["purchase"], normalize="index").round(4))

fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    sns.countplot(x=col, hue="purchase", data=data, ax=ax)
    ax.set_title(f"{col} by purchase")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
binary_cols = ["discount_seen", "ad_clicked", "returning_user"]
for col in binary_cols:
    print(f"\n{col} vs purchase (row proportions):")
    display(pd.crosstab(data[col], data["purchase"], normalize="index").round(4))

fig, axes = plt.subplots(1, len(binary_cols), figsize=(5 * len(binary_cols), 4))
for ax, col in zip(axes, binary_cols):
    sns.countplot(x=col, hue="purchase", data=data, ax=ax)
    ax.set_title(f"{col} by purchase")

plt.tight_layout()
plt.show()

## 6. Numerical features vs purchase

In [ ]:
key_numeric = ["bounce_rate", "time_on_site", "pages_viewed", "cart_items", "previous_purchases"]

print("Mean values by purchase class:")
display(data.groupby("purchase")[key_numeric].mean().round(2))

n_cols = 3
n_rows = math.ceil(len(key_numeric) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for i, col in enumerate(key_numeric):
    sns.boxplot(x="purchase", y=col, data=data, ax=axes[i])
    axes[i].set_title(f"{col} by purchase")

for j in range(len(key_numeric), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=data,
    x="time_on_site",
    y="pages_viewed",
    hue="purchase",
    alpha=0.5,
    palette="Set1",
)
plt.title("Time on site vs pages viewed (colored by purchase)")
plt.tight_layout()
plt.show()

## 7. Multivariate analysis

In [ ]:
corr_cols = [c for c in feature_cols]
corr_matrix = data[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numerical features)")
plt.tight_layout()
plt.show()

if "purchase" in data.columns:
    purchase_corr = data[corr_cols + ["purchase"]].corr()["purchase"].drop("purchase").sort_values(key=abs, ascending=False)
    print("\nCorrelation with purchase:")
    display(purchase_corr.to_frame("corr_with_purchase"))

In [ ]:
pairplot_cols = ["age", "bounce_rate", "time_on_site", "cart_items", "purchase"]
sns.pairplot(data[pairplot_cols].dropna(), hue="purchase", diag_kind="hist", corner=True)
plt.suptitle("Pairwise relationships (selected features)", y=1.02)
plt.show()

## 8. Business questions

In [ ]:
questions = {
    "Returning users — purchase rate": data.groupby("returning_user")["purchase"].mean(),
    "Discount seen — purchase rate": data.groupby("discount_seen")["purchase"].mean(),
    "Ad clicked — purchase rate": data.groupby("ad_clicked")["purchase"].mean(),
    "Device type — purchase rate": data.groupby("device_type")["purchase"].mean(),
    "Gender — purchase rate": data.groupby("gender")["purchase"].mean(),
}

for title, series in questions.items():
    print(f"\n{title}:")
    print(series.round(4))

print("\nAvg session metrics by device:")
display(
    data.groupby("device_type")[["time_on_site", "avg_session_time", "bounce_rate", "pages_viewed"]]
    .mean()
    .round(2)
)

print("\nEngagement by returning_user:")
display(
    data.groupby("returning_user")[["previous_purchases", "cart_items", "pages_viewed"]]
    .mean()
    .round(2)
)

## 9. Summary & preprocessing plan

In [ ]:
n_missing = data.isnull().sum().sum()
n_no_purchase = (data["purchase"] == 0).sum()
n_purchase = (data["purchase"] == 1).sum()
high_corr = corr_matrix["time_on_site"].get("avg_session_time", np.nan)

summary = f"""
Dataset: e-commerce user behavior ({data.shape[0]} rows, {data.shape[1]} columns)

Key findings:
- Missing values: {n_missing} total across all cells; main gaps in ad_clicked and related rows.
- Target imbalance: purchase=1 → {n_purchase} rows ({n_purchase/len(data)*100:.2f}%), purchase=0 → {n_no_purchase} rows ({n_no_purchase/len(data)*100:.2f}%).
- user_id is an identifier ({data['user_id'].nunique()} unique values) — drop for modeling.
- time_on_site vs avg_session_time correlation: {high_corr:.2f} — consider using only one.

Preprocessing plan:
1. Drop or exclude user_id from features.
2. Impute ad_clicked (mode) or drop rows with missing target/features.
3. One-hot encode gender and device_type.
4. Scale numerical features for neural networks.
5. Handle class imbalance (class weights, oversampling minority, or stratified split).
6. Train/validation/test split with stratification on purchase.
"""

print(summary)